# CSA0812 – Assignment 5
## A3 – Design the Analysis and Visualisation

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Outcome:** CO5  
**Marks:** 5 | K4 – Analyse

**Purpose:** This notebook is organised as a separate submission for this rubric item. Written explanations are provided in text cells. Where implementation is required, the Python cells can be run in Google Colab.

## A3.1 – Choose the Charts (3 Marks)

| Board question | Chart | Reason |
|---|---|---|
| Trend of PM2.5 over a month | Line chart | Date is ordered time data. A line makes rises, falls and peaks visible across the month. Use daily mean PM2.5 for one station or separate lines for selected stations. |
| Comparison of stations | Bar chart | Each station is a category. Bars make average or median PM2.5 easy to compare and rank. Label station names and the aggregation period. |
| Relationship between temperature and PM2.5 | Scatter plot | Each point represents one paired reading. A pattern may show positive, negative or no association without implying causation. |
| Hours with the worst air | Bar chart of mean PM2.5 by hour, optionally supported by a box plot | A bar chart ranks average PM2.5 across hours 0–23. A box plot additionally displays spread and outliers for each hour. |

**Chart design choices:** Every chart needs a descriptive title, labelled axes, units, a clearly stated date range and a legend when more than one station is shown. Missing or imputed values should be identified so they are not mistaken for original sensor measurements.

## A3.2 – Design the Summary Table (2 Marks)

The daily summary table is grouped by station and date.

| Column | Meaning |
|---|---|
| station_id / station_name | Monitoring station |
| observation_date | Day being summarised |
| pm25_min | Minimum valid PM2.5 reading for the day |
| pm25_max | Maximum valid PM2.5 reading for the day |
| pm25_mean | Mean valid PM2.5 reading for the day |
| valid_reading_count | Number of non-NULL valid readings included |

A row is produced for each station/date pair. Invalid and missing readings are excluded from the aggregate; the count makes the amount of valid evidence visible. The table can be generated with SQL `GROUP BY station_id, observation_date` using `MIN`, `MAX`, `AVG` and `COUNT(pm25)`. If a day has no valid PM2.5 readings, the count is zero and the mean is NULL.

### Example SQL used to create the daily summary

```sql
SELECT
    station_id,
    observation_date,
    MIN(pm25) AS pm25_min,
    MAX(pm25) AS pm25_max,
    AVG(pm25) AS pm25_mean,
    COUNT(pm25) AS valid_reading_count
FROM readings
GROUP BY station_id, observation_date
ORDER BY station_id, observation_date;
```

This aggregation lets the database calculate the summary close to where the data is stored. The result can be exported to CSV or used directly by the charting program.

## Alternative Concept and Complexity Analysis

If the summary were computed using Python loops after reading all rows, the program would scan each reading, use a dictionary keyed by `(station, day)`, and maintain minimum, maximum, running sum and count.

| Approach | Time | Extra working space |
|---|---|---|
| SQL `GROUP BY` | Typically O(n) scan/aggregation; sorting or indexes can affect actual plan | O(g) aggregation state for g station/day groups, plus database engine working memory |
| Python loop with dictionary | O(n) average when each row is processed once and dictionary operations are O(1) average | O(g) for per-group min/max/sum/count; O(n) if all raw rows are first retained in memory |

A streaming Python loop does not need to hold all readings. The assignment's comparison is about the aggregation method: both need to examine n observations, but SQL can aggregate inside the database and avoid transferring every raw row into Python. The result should be checked against the same grouping keys and valid-value rules.

## A3 – Conclusion

A line chart is best for time trends, a bar chart for station and hourly comparisons, and a scatter plot for temperature/PM2.5 association. The daily summary table should report minimum, maximum, mean and valid count per station/date. SQL `GROUP BY` is a good choice when readings already reside in SQLite.